# GCN Kafka → ELK

Streams NASA GCN alerts into a local Elastic stack and gives you a Kibana view over them.

```
kafka.gcn.nasa.gov  ──▶  Spark Structured Streaming  ──▶  Elasticsearch  ──▶  Kibana
   (SASL_SSL /             read + decode + bulk index      gcn-<topic>       Discover /
    OAUTHBEARER)           (this is the "L" in ELK)        indices           dashboards
```

Logstash is deliberately not in the stack: Spark already holds the GCN OAuth credentials and
the exactly-once checkpointing, so it does the collect/transform/ship job itself and writes
straight to the Elasticsearch `_bulk` API.

**Delivery guarantees.** There is no event-time watermark here, and none is wanted: the
pipeline is stateless, so no record is ever dropped for arriving late. Completeness rests on
three things instead — Spark's offset checkpoint (at-least-once), deterministic
`topic-partition-offset` document IDs that turn a replay into an overwrite (effectively
exactly-once in Elasticsearch), and a dead-letter index that stops a rejected document from
being silently acknowledged. Section 6b audits the result against Kafka's own offset sequence.

`gcn.heartbeat` (one tick a second) is consumed but never indexed: it is the endpoint
liveness probe. Only an *abnormal* heartbeat — silence beyond a threshold, and the eventual
recovery — is written to Elasticsearch, so the index holds incidents, not ticks.

Note that `@timestamp` is the Kafka broker append time, not the astrophysical event time inside
the payload — those live under `alert` and differ per topic schema.

**Prerequisites**
- Docker Desktop running, with ~4 GB free for Elasticsearch + Kibana.
- `GCN_CLIENT_ID` / `GCN_CLIENT_SECRET` in `.env` (same credentials as `sample-pyspark.ipynb`).
- The `gcn` kernel, with `pyspark` and `python-dotenv` installed. Everything else here is stdlib.

**Local dev only:** the Elasticsearch below runs with security disabled and no TLS. Do not
expose those ports beyond localhost.

In [21]:
import base64
import json
import os
import re
import shutil
import ssl
import subprocess
import tempfile
import time
import urllib.error
import urllib.request
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# --- kernel environment ----------------------------------------------------
# VS Code starts the Jupyter kernel from launchd rather than a login shell, so
# the kernel inherits a bare PATH and two things break under "Run All" that are
# fine in a terminal:
#   * docker lives in /usr/local/bin, which that PATH doesn't include;
#   * java falls through to the /usr/bin/java stub, which picks the *newest*
#     installed JDK. Spark 4.x supports Java 17 and 21 only -- on anything newer
#     it aborts with ClassNotFoundException: jdk.internal.ref.Cleaner.
# Resolving both here makes the notebook behave the same however it was started.

DOCKER = shutil.which("docker") or next(
    (
        candidate
        for candidate in (
            "/usr/local/bin/docker",
            "/opt/homebrew/bin/docker",
            str(Path.home() / ".docker" / "bin" / "docker"),
            "/Applications/Docker.app/Contents/Resources/bin/docker",
        )
        if Path(candidate).exists()
    ),
    None,
)


def _jdk_major(java_home):
    """Major version of the JDK at java_home, or None if it isn't usable."""
    java = Path(java_home or "") / "bin" / "java"
    if not java.is_file():
        return None
    probe = subprocess.run([str(java), "-version"], capture_output=True, text=True)
    found = re.search(r'version "(\d+)', probe.stderr)
    return int(found.group(1)) if found else None


SPARK_JDKS = (21, 17)
if _jdk_major(os.environ.get("JAVA_HOME")) not in SPARK_JDKS:
    for version in SPARK_JDKS:
        located = subprocess.run(
            ["/usr/libexec/java_home", "-v", str(version)],
            capture_output=True,
            text=True,
        )
        if located.returncode == 0:
            os.environ["JAVA_HOME"] = located.stdout.strip()
            # Spark reads JAVA_HOME, but keep PATH consistent for anything that
            # shells out to `java` directly.
            os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ.get('PATH', '')}"
            break


PROJECT = Path.cwd()
COMPOSE_FILE = PROJECT / "docker-compose.elk.yml"
CHECKPOINT_DIR = PROJECT / ".checkpoints" / "gcn-elk"

STACK_VERSION = os.environ.get("ELASTIC_STACK_VERSION", "9.5.3")
ES_URL = os.environ.get("ES_URL", "http://localhost:9200").rstrip("/")
KIBANA_URL = os.environ.get("KIBANA_URL", "http://localhost:5601").rstrip("/")

# es01 runs with Elasticsearch security defaults on: HTTPS plus basic auth.
# ES_CA_CERT points at the CA extracted from the container so TLS is verified
# properly rather than waved through with an unverified context.
ES_USER = os.environ.get("ES_USER", "elastic")
ES_PASSWORD = os.environ.get("ES_PASSWORD") or None
ES_CA_CERT = os.environ.get("ES_CA_CERT") or None

CLIENT_ID = os.environ["GCN_CLIENT_ID"]
CLIENT_SECRET = os.environ["GCN_CLIENT_SECRET"]

TOPIC_LIST = [
    "gcn.notices.boom.alert",
    "gcn.circulars",
    "gcn.notices.dsa110.frb",
    "igwn.gwalert",
    "gcn.notices.maxi.gsc.known",
    "gcn.notices.maxi.gsc.unknown",
    "gcn.notices.superk.sn_alert",
    "gcn.notices.swift.bat.guano",
]
TOPICS = ",".join(TOPIC_LIST)

# gcn.heartbeat ticks once a second. It is read only to prove the GCN endpoint
# is alive while the science topics stream -- ticks are never indexed. If no tick
# arrives for HEARTBEAT_MAX_AGE seconds the outage (and later the recovery) is
# written to HEARTBEAT_INDEX, so the index holds incidents rather than noise.
HEARTBEAT_TOPIC = "gcn.heartbeat"
HEARTBEAT_MAX_AGE = 30
HEARTBEAT_INDEX = "gcn-heartbeat-events"

# --- completeness controls -------------------------------------------------
# True  -> the stream fails loudly if Kafka has already aged out offsets we
#          still owe (GCN retains roughly the last few days).
# False -> Spark silently skips ahead to the earliest surviving offset and only
#          logs a WARN. Convenient for demos, unsafe for anything archival.
FAIL_ON_DATA_LOSS = True

# Records that Elasticsearch rejects are parked here instead of being dropped.
DLQ_INDEX = "gcn-dead-letter"

# Total records per trigger across ALL partitions of all topics, not per topic.
# Sized by bytes, not rows: an igwn.gwalert record carries a base64 skymap and
# runs to ~3 MB, and GCN serves them at roughly 4 s per record. A batch must
# finish well inside STREAM_SECONDS or it is cancelled and its reads wasted.
MAX_OFFSETS_PER_TRIGGER = 20

# Flush a _bulk request once its payload reaches this many bytes. Elasticsearch
# rejects any request over http.max_content_length (100 MB by default).
BULK_MAX_BYTES = 20 * 1024 * 1024

# Documents older than this are pruned from Elasticsearch (section 6d). Kafka's
# own retention is roughly comparable for the science topics, so this bounds
# index growth rather than discarding anything still replayable.
RETENTION_DAYS = 60

# How long to let the stream run before stopping it, in seconds.
STREAM_SECONDS = 600

# After the deadline, wait up to this long for the in-flight batch to commit
# before stopping. stop() cancels a running batch outright, which discards
# everything it has read so far and leaves the checkpoint uncommitted.
DRAIN_SECONDS = 300

print(f"stack {STACK_VERSION} -> {ES_URL} / {KIBANA_URL}")
print(f"{len(TOPIC_LIST)} science topics, checkpoint at {CHECKPOINT_DIR}")
print(f"failOnDataLoss={FAIL_ON_DATA_LOSS}, dead-letter index={DLQ_INDEX}")
print(f"{MAX_OFFSETS_PER_TRIGGER} records/trigger, bulk flush at {BULK_MAX_BYTES >> 20} MB")
print(f"pruning documents older than {RETENTION_DAYS} days")
print(f"heartbeat: {HEARTBEAT_TOPIC}, stale after {HEARTBEAT_MAX_AGE}s, incidents -> {HEARTBEAT_INDEX}")
print(f"docker={DOCKER}, JAVA_HOME={os.environ.get('JAVA_HOME')}" f" (Java {_jdk_major(os.environ.get('JAVA_HOME'))})")

stack 9.5.3 -> https://localhost:9200 / http://localhost:5601
8 science topics, checkpoint at /Users/harrywatson/Dropbox/Coding/GCN/.checkpoints/gcn-elk
failOnDataLoss=True, dead-letter index=gcn-dead-letter
20 records/trigger, bulk flush at 20 MB
pruning documents older than 60 days
heartbeat: gcn.heartbeat, stale after 30s, incidents -> gcn-heartbeat-events
docker=/usr/local/bin/docker, JAVA_HOME=/opt/homebrew/Cellar/openjdk@21/21.0.12.1/libexec/openjdk.jdk/Contents/Home (Java 21)


## 1. Bring up Elasticsearch + Kibana

Reuses the `es01` / `kib01` containers already running on this machine rather than starting a
second stack. Those run with Elasticsearch security defaults, so the cluster is reachable over
**HTTPS with basic auth** on `:9200` -- set `ES_USER` / `ES_PASSWORD` in `.env`, and point
`ES_CA_CERT` at the CA copied out of the container:

```
docker exec es01 cat /usr/share/elasticsearch/config/certs/http_ca.crt > certs/es01_ca.crt
```

`docker-compose.elk.yml` remains as a self-contained alternative -- set `MANAGE_STACK = True`
below to use it. It binds 9201/5602 so it can run alongside `es01` without a port clash.

In [22]:
if DOCKER is None:
    raise RuntimeError(
        "docker CLI not found. It is usually /usr/local/bin/docker; if the kernel "
        "cannot see it, add that directory to PATH or set DOCKER explicitly above."
    )

# es01 / kib01 are managed outside this notebook, so nothing is started here.
# Flip to True to run the self-contained stack in docker-compose.elk.yml instead
# (it binds 9201/5602 to stay clear of es01 -- point ES_URL / KIBANA_URL at it).
MANAGE_STACK = False

if MANAGE_STACK:
    up = subprocess.run(
        [DOCKER, "compose", "-f", str(COMPOSE_FILE), "up", "-d"],
        capture_output=True,
        text=True,
    )
    print(up.stdout or up.stderr)
    up.check_returncode()
else:
    running = subprocess.run(
        [DOCKER, "ps", "--format", "{{.Names}}"],
        capture_output=True,
        text=True,
    ).stdout.split()
    missing = [name for name in ("es01", "kib01") if name not in running]
    if missing:
        raise RuntimeError(
            f"expected containers not running: {', '.join(missing)}. "
            "Start them, or set MANAGE_STACK = True to use the bundled compose stack."
        )
    print("reusing es01 / kib01")

reusing es01 / kib01


In [23]:
if ES_CA_CERT and not Path(ES_CA_CERT).exists():
    raise FileNotFoundError(
        f"ES_CA_CERT={ES_CA_CERT} not found. Extract it with:\n"
        "  docker exec es01 cat /usr/share/elasticsearch/config/certs/http_ca.crt"
        f" > {ES_CA_CERT}"
    )

SSL_CONTEXT = ssl.create_default_context(cafile=ES_CA_CERT) if ES_CA_CERT else None

if ES_PASSWORD:
    _token = base64.b64encode(f"{ES_USER}:{ES_PASSWORD}".encode()).decode()
    AUTH_HEADER = {"Authorization": f"Basic {_token}"}
else:
    AUTH_HEADER = {}


def es(method, path, body=None, base=None, headers=None):
    """Minimal JSON client for Elasticsearch/Kibana. Returns (status, parsed_body)."""
    url = f"{base or ES_URL}{path}"
    data = None
    hdrs = {"Content-Type": "application/json", **AUTH_HEADER}
    if body is not None:
        data = body.encode() if isinstance(body, str) else json.dumps(body).encode()
    if headers:
        hdrs.update(headers)
    req = urllib.request.Request(url, data=data, method=method, headers=hdrs)
    try:
        ctx = SSL_CONTEXT if url.startswith("https") else None
        with urllib.request.urlopen(req, timeout=60, context=ctx) as resp:
            raw = resp.read().decode()
            return resp.status, (json.loads(raw) if raw.strip().startswith(("{", "[")) else raw)
    except urllib.error.HTTPError as exc:
        raw = exc.read().decode()
        try:
            return exc.code, json.loads(raw)
        except json.JSONDecodeError:
            return exc.code, raw


def wait_for(name, check, timeout=300, interval=5):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if check():
                print(f"{name}: ready")
                return True
        except Exception:
            pass
        time.sleep(interval)
    raise TimeoutError(f"{name} not ready after {timeout}s")


wait_for(
    "elasticsearch",
    lambda: es("GET", "/_cluster/health")[1].get("status") in ("green", "yellow"),
)
wait_for(
    "kibana",
    lambda: es("GET", "/api/status", base=KIBANA_URL)[1]["status"]["overall"]["level"]
    == "available",
)

print(json.dumps(es("GET", "/")[1]["version"], indent=2))

elasticsearch: ready
kibana: ready
{
  "number": "9.4.2",
  "build_flavor": "default",
  "build_type": "docker",
  "build_hash": "c402c2b36d90eae29c0182f86bd9050fd0b746cc",
  "build_date": "2026-05-25T22:10:36.017759931Z",
  "build_snapshot": false,
  "lucene_version": "10.4.0",
  "minimum_wire_compatibility_version": "8.19.0",
  "minimum_index_compatibility_version": "8.0.0"
}


## 2. Index templates

One index per topic (`gcn-<topic>`) so that topics with clashing field types can't break each
other's mappings. Strings map to `keyword` with an analysed `.text` sub-field, which keeps
Kibana aggregations cheap while still allowing free-text search on circular bodies.

A second, higher-priority template covers the dead-letter index. It is `dynamic: false`, so a
document that Elasticsearch rejected for a mapping reason can always be parked there — the
whole point is that this index never rejects anything.

In [24]:
TEMPLATE = {
    "index_patterns": ["gcn-*"],
    "priority": 500,
    "template": {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "index.mapping.total_fields.limit": 4000,
            "index.mapping.ignore_malformed": True,
        },
        "mappings": {
            "dynamic_templates": [
                {
                    "strings": {
                        "match_mapping_type": "string",
                        "mapping": {
                            "type": "keyword",
                            "ignore_above": 512,
                            "fields": {"text": {"type": "text"}},
                        },
                    }
                }
            ],
            "properties": {
                "@timestamp": {"type": "date"},
                "topic": {"type": "keyword"},
                "ingested_at": {"type": "date"},
                "kafka": {
                    "properties": {
                        "partition": {"type": "integer"},
                        "offset": {"type": "long"},
                        "key": {"type": "keyword"},
                    }
                },
                "payload": {"type": "text", "index": False},
                "parse_error": {"type": "keyword"},
                # Long strings must not fall through to the keyword dynamic
                # template: past ignore_above the keyword copy is dropped into
                # _ignored, and Kibana's fields API then returns nothing, so the
                # field shows as empty in Discover. Name the known offenders.
                "alert": {
                    "properties": {
                        # gcn.circulars: the full circular text.
                        "body": {"type": "text"},
                        # igwn.gwalert: base64 FITS skymaps, 1-2 MB each. Kept
                        # in _source for retrieval, never indexed.
                        "event": {
                            "properties": {
                                "skymap": {"type": "keyword", "index": False, "doc_values": False},
                            }
                        },
                        "external_coinc": {
                            "properties": {
                                "combined_skymap": {"type": "keyword", "index": False, "doc_values": False},
                            }
                        },
                    }
                },
            },
        },
    },
}

# Higher priority so it wins over "gcn-*" for the dead-letter index itself.
DLQ_TEMPLATE = {
    "index_patterns": [f"{DLQ_INDEX}*"],
    "priority": 600,
    "template": {
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {
            # Never reject: unknown fields are kept in _source but not indexed.
            "dynamic": False,
            "properties": {
                "@timestamp": {"type": "date"},
                "topic": {"type": "keyword"},
                "ingested_at": {"type": "date"},
                "batch_id": {"type": "long"},
                "kafka": {
                    "properties": {
                        "partition": {"type": "integer"},
                        "offset": {"type": "long"},
                        "key": {"type": "keyword"},
                    }
                },
                "error": {
                    "properties": {
                        "type": {"type": "keyword"},
                        "reason": {"type": "text"},
                        "status": {"type": "integer"},
                    }
                },
                "payload": {"type": "text", "index": False},
            },
        },
    },
}

for name, body in (("gcn", TEMPLATE), ("gcn-dead-letter", DLQ_TEMPLATE)):
    status, resp = es("PUT", f"/_index_template/{name}", body)
    print(name, status, resp)

gcn 200 {'acknowledged': True}
gcn-dead-letter 200 {'acknowledged': True}


## 3. Read the GCN stream

Same OAUTHBEARER handshake as `sample-pyspark.ipynb`. The session timezone is pinned to UTC so
the Kafka record timestamps serialise as UTC instants for Elasticsearch.

Two readers share the connection settings. The science topics read from `earliest` under the
durable checkpoint. The heartbeat reader starts at `latest` with no durable checkpoint: a
backlog of old ticks says nothing about whether the endpoint is up *now*, and replaying it
would only delay the first real reading.

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder.config(
        "spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    # local[*] alone aborts the whole run on a single task failure; allow a
    # Kafka fetch timeout to be retried before the batch is given up on.
    .master("local[*, 3]")
    # In local mode this one JVM is driver and executor. The 1 GB default heap
    # has to hold the Kafka fetch buffers, each batch's rows (a gwalert batch
    # is tens of MB of Java strings, copied again when pickled for Python) and
    # the UI's history of every job and query plan -- it ran out after ~100
    # batches. Give it room and keep the UI history short.
    .config("spark.driver.memory", "4g")
    .config("spark.ui.retainedJobs", "200")
    .config("spark.ui.retainedStages", "200")
    .config("spark.sql.ui.retainedExecutions", "100")
    # The console progress bar redraws "[Stage 74:> (0 + 1) / 1]" over the
    # cell output with carriage returns, which in a notebook leaves stray
    # bars and trailing blank lines interleaved with our own prints.
    .config("spark.ui.showConsoleProgress", "false")
    # These, like master, are JVM launch settings: they take effect only when
    # the kernel is restarted, not on re-running this cell.
    .config("spark.sql.session.timeZone", "UTC")
    .appName("gcn_elk")
    .getOrCreate()
)

# --- log noise -------------------------------------------------------------
# Spark logs through log4j2. Two of its loggers say nothing useful here and
# would otherwise land in the middle of the per-batch lines from section 5:
#
#   * ProcessingTimeExecutor WARNs once per batch that the batch outran the 10 s
#     trigger. With maxOffsetsPerTrigger set that is the *designed* behaviour
#     while a backlog drains -- Spark just starts the next batch immediately --
#     and the batch line now prints the duration anyway.
#   * DAGScheduler WARNs "Failed to cancel job group ... Cannot find active
#     jobs" at shutdown, because stop() cancels job groups whose jobs have
#     already finished. Silenced only around the stop itself (section 5).
#
# Raising two named loggers beats setLogLevel("ERROR") on the whole of Spark,
# which would also hide the warnings worth reading.
def set_log_level(logger, level="ERROR"):
    """Set one JVM logger's level; returns the level it had before, or None.

    log4j's internals are not a stable API across Spark builds, so a failure
    here only costs the quieting -- never the run it was tidying the output of.
    """
    try:
        log4j = spark._jvm.org.apache.logging.log4j
        previous = log4j.LogManager.getLogger(logger).getLevel().toString()
        log4j.core.config.Configurator.setLevel(logger, log4j.Level.toLevel(level))
        return previous
    except Exception as exc:
        print(f"  note: could not set log level for {logger} ({type(exc).__name__})")
        return None


@contextmanager
def quiet(*loggers, level="ERROR"):
    """Raise some loggers' threshold for the duration of the block."""
    restore = {name: set_log_level(name, level) for name in loggers}
    try:
        yield
    finally:
        for name, level_before in restore.items():
            if level_before:
                set_log_level(name, level_before)


# Spark moved this class to ...streaming.runtime in 4.1; naming a logger that
# does not exist is harmless, so cover both and stay version-agnostic.
for _noisy in (
    "org.apache.spark.sql.execution.streaming.ProcessingTimeExecutor",
    "org.apache.spark.sql.execution.streaming.runtime.ProcessingTimeExecutor",
):
    set_log_level(_noisy)

GCN_KAFKA = {
    "kafka.bootstrap.servers": "kafka.gcn.nasa.gov:9092",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "OAUTHBEARER",
    # kafka-clients 3.x path. If 4.2.0 bundles kafka-clients 4.x, drop
    # ".secured" — verify with the unzip check against the resolved JAR.
    "kafka.sasl.login.callback.handler.class": (
        "org.apache.kafka.common.security.oauthbearer.secured.OAuthBearerLoginCallbackHandler"
    ),
    "kafka.sasl.jaas.config": (
        "org.apache.kafka.common.security.oauthbearer.OAuthBearerLoginModule required "
        f'clientId="{CLIENT_ID}" clientSecret="{CLIENT_SECRET}";'
    ),
    "kafka.sasl.oauthbearer.token.endpoint.url": "https://auth.gcn.nasa.gov/oauth2/token",
    # Throughput over a long-haul link. The client's default 64 KB socket
    # receive buffer caps one TCP stream at 64 KB per round trip -- about
    # 300 KB/s at 250 ms RTT, i.e. ~4 s per 1 MB gwalert record. -1 lets the
    # OS autotune the window. Fetch sizes stay modest: the broker appears to
    # throttle around 500 KB/s per client, and Kafka throttles by *delaying*
    # the response, so an oversized fetch can stall past Spark's 120 s poll
    # timeout. 4 MB still fits the largest (~3 MB) gwalert record whole.
    "kafka.receive.buffer.bytes": -1,
    "kafka.max.partition.fetch.bytes": 4 * 1024 * 1024,
    "kafka.fetch.max.bytes": 8 * 1024 * 1024,
}


def gcn_stream(topics, **options):
    """A decoded GCN stream over `topics`, with per-reader Kafka source options."""
    reader = spark.readStream.format("kafka").options(**GCN_KAFKA).option("subscribe", topics)
    for name, value in options.items():
        reader = reader.option(name, value)
    return reader.load().select(
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp"),
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("payload"),
    )


# startingOffsets applies only while the checkpoint directory is empty; once a
# batch has committed, the offset log decides where the stream resumes and this
# is ignored. So "earliest" means "from the start of retention on a fresh
# checkpoint", which is what deleting .checkpoints/gcn-elk asks for.
#
# (It briefly held an explicit offset map instead: the checkpoint predating the
# heartbeat probe's split carried gcn.heartbeat, whose aged-out offsets failed
# every restart. That checkpoint is parked at .checkpoints/gcn-elk.heartbeat-
# poisoned, and the live one has long since committed past the offsets it
# pinned, so the override had no effect on any run after the first.)
decoded = gcn_stream(
    TOPICS,
    startingOffsets="earliest",
    maxOffsetsPerTrigger=MAX_OFFSETS_PER_TRIGGER,
    failOnDataLoss=str(FAIL_ON_DATA_LOSS).lower(),
)

# Liveness only: start at the newest tick, and never fail over aged-out offsets.
heartbeat = gcn_stream(
    HEARTBEAT_TOPIC, startingOffsets="latest", failOnDataLoss="false"
)

decoded.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- key: string (nullable = true)
 |-- payload: string (nullable = true)



## 4. Bulk sink

`foreachBatch` gives us the batch as a normal DataFrame, so each micro-batch becomes one
`_bulk` request. Document IDs are `topic-partition-offset`, which makes the write idempotent:
replaying a checkpoint overwrites documents instead of duplicating them.

**Durability contract.** A batch may only commit its offsets once every record in it is
durably somewhere. So rejected documents are re-sent to the dead-letter index, and if *that*
write fails the function raises — the batch does not commit, and the next run replays the same
offset range. Nothing is acknowledged that isn't stored.

**Heartbeat sink.** The heartbeat batch only advances `HEARTBEAT["last_seen"]`. Section 5
compares that against the wall clock: silence beyond `HEARTBEAT_MAX_AGE` is written to
`HEARTBEAT_INDEX` as a `lost` event, and the next tick after that writes a matching `restored`
event carrying the measured gap. A healthy run writes nothing.

In [26]:
def index_for(topic):
    # gcn.circulars -> gcn-circulars, igwn.gwalert -> gcn-igwn-gwalert.
    # The gcn- prefix is what the index template matches on, so non-gcn topics
    # need it added; gcn.* topics already carry it and must not be doubled.
    slug = topic.lower().replace(".", "-").replace("_", "-")
    return slug if slug.startswith("gcn-") else f"gcn-{slug}"


def event_time(ts):
    # PySpark hands back a *naive* datetime in the Python process's local zone
    # (spark.sql.session.timeZone does not apply to collect()). Serialised as-is
    # Elasticsearch reads it as UTC, shifting every @timestamp by the local offset.
    return ts.astimezone(timezone.utc).isoformat() if ts else None


def to_document(row):
    doc = {
        "@timestamp": event_time(row["timestamp"]),
        "ingested_at": datetime.now(timezone.utc).isoformat(),
        "topic": row["topic"],
        "kafka": {
            "partition": row["partition"],
            "offset": row["offset"],
            "key": row["key"],
        },
    }
    payload = row["payload"]
    if payload is None:
        doc["parse_error"] = "empty_payload"
        return doc
    try:
        parsed = json.loads(payload)
    except (json.JSONDecodeError, TypeError):
        doc["payload"] = payload
        doc["parse_error"] = "not_json"
        return doc
    doc["alert"] = parsed if isinstance(parsed, dict) else {"value": parsed}
    return doc


def bulk(lines):
    """POST an NDJSON bulk body. Returns the list of per-item results."""
    status, body = es("POST", "/_bulk?refresh=false", "\n".join(lines) + "\n")
    if status >= 300:
        raise RuntimeError(f"bulk request failed ({status}): {body}")
    return body.get("items", [])


def to_dead_letter(row, doc_id, error, batch_id):
    """Park a rejected record with enough context to replay it by hand."""
    return {
        "@timestamp": event_time(row["timestamp"]),
        "ingested_at": datetime.now(timezone.utc).isoformat(),
        "batch_id": batch_id,
        "topic": row["topic"],
        "kafka": {
            "partition": row["partition"],
            "offset": row["offset"],
            "key": row["key"],
        },
        "error": {
            "type": error.get("type"),
            "reason": error.get("reason"),
            "status": error.get("status"),
        },
        # Always the raw bytes as received, so nothing depends on our parsing.
        "payload": row["payload"],
    }


# --- JVM-side failures -----------------------------------------------------
# When a batch fails, Spark matches the Python traceback against the regex
#   py4j.protocol.Py4JJavaError: An error occurred while calling((.|\r\n|\r|\n)*)(...)
# to decide whether the query was merely interrupted. That group recurses once
# per character in Java's regex engine, and a traceback carrying a Java stack
# trace (a cancelled batch, say) overflows the stream thread -- which can take
# the whole SparkContext down. So a JVM-originated error is re-raised short,
# with its chain suppressed: the batch still fails and is never committed, but
# the JVM never sees a traceback it can choke on.
def is_jvm_error(exc):
    return type(exc).__module__.split(".")[0] in ("py4j", "pyspark")


def reraise_short(batch_id, exc):
    lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
    # The innermost "Caused by:" is the root cause; failing that, the Java
    # exception summary on Py4JJavaError's second line; failing that, line 1.
    causes = [line[len("Caused by:"):].strip() for line in lines if line.startswith("Caused by:")]
    summary = causes[-1] if causes else next(
        (line.lstrip(": ") for line in lines if line.startswith(":")), None
    )
    summary = summary or (lines[0] if lines else type(exc).__name__)
    raise RuntimeError(f"batch {batch_id} aborted: {summary[:300]}") from None


# --- heartbeat -------------------------------------------------------------
HEARTBEAT = {"last_seen": None, "ticks": 0, "lost_at": None}


def heartbeat_age():
    """Seconds since the newest tick, or None if none has arrived yet."""
    if HEARTBEAT["last_seen"] is None:
        return None
    return (datetime.now(timezone.utc) - HEARTBEAT["last_seen"]).total_seconds()


def record_heartbeat_event(event, **detail):
    """Write one abnormal-heartbeat document. Normal ticks never reach here."""
    now = datetime.now(timezone.utc)
    doc = {"@timestamp": now.isoformat(), "topic": HEARTBEAT_TOPIC, "event": event, **detail}
    doc_id = f"{event}-{now.strftime('%Y%m%dT%H%M%S')}"
    status, body = es("PUT", f"/{HEARTBEAT_INDEX}/_doc/{doc_id}?refresh=true", doc)
    if status >= 300:
        raise RuntimeError(f"could not record heartbeat {event} ({status}): {body}")
    print(f"  ⚠  heartbeat {event}: " + ", ".join(f"{k}={v}" for k, v in detail.items()))


def check_heartbeat():
    """Called from the driver loop. Flags the endpoint once when it goes quiet."""
    age = heartbeat_age()
    if age is None or age <= HEARTBEAT_MAX_AGE or HEARTBEAT["lost_at"] is not None:
        return
    HEARTBEAT["lost_at"] = datetime.now(timezone.utc)
    record_heartbeat_event(
        "lost", last_seen=HEARTBEAT["last_seen"].isoformat(), silent_for_s=round(age)
    )


def track_heartbeat(batch_df, batch_id):
    try:
        newest = batch_df.agg({"timestamp": "max"}).first()[0]
        ticks = batch_df.count() if newest is not None else 0
    except Exception as exc:
        if is_jvm_error(exc):
            reraise_short(batch_id, exc)
        raise
    if newest is None:
        return
    newest = newest.astimezone(timezone.utc)
    HEARTBEAT["ticks"] += ticks
    previous = HEARTBEAT["last_seen"]
    HEARTBEAT["last_seen"] = max(newest, previous) if previous else newest
    if HEARTBEAT["lost_at"] is not None:
        # First tick after an outage: close it out with the gap actually observed.
        gap = (newest - previous).total_seconds() if previous else None
        record_heartbeat_event(
            "restored", lost_at=HEARTBEAT["lost_at"].isoformat(), gap_s=round(gap) if gap else None
        )
        HEARTBEAT["lost_at"] = None


# --- science topics --------------------------------------------------------
# What each batch wrote, for the one-line-per-batch report in section 5. The
# sink cannot print it itself: the lag and duration it belongs next to are only
# known once the batch has committed, which is after foreachBatch returns. The
# driver loop pops each entry as it reports it, so this never grows.
BATCH_STATS = {}


def doc_id_for(row):
    return f"{row['topic']}-{row['partition']}-{row['offset']}"


def index_chunk(rows, batch_id):
    """One _bulk request for `rows`. Returns how many were dead-lettered."""
    lines = []
    for row in rows:
        lines.append(json.dumps({"index": {"_index": index_for(row["topic"]), "_id": doc_id_for(row)}}))
        lines.append(json.dumps(to_document(row), default=str))

    items = bulk(lines)

    rejected = [
        (rows[i], item.get("index", {}))
        for i, item in enumerate(items)
        if item.get("index", {}).get("status", 200) >= 300
    ]
    if not rejected:
        return 0

    dlq_lines = []
    for row, result in rejected:
        error = dict(result.get("error") or {})
        error["status"] = result.get("status")
        dlq_lines.append(json.dumps({"index": {"_index": DLQ_INDEX, "_id": doc_id_for(row)}}))
        dlq_lines.append(json.dumps(to_dead_letter(row, doc_id_for(row), error, batch_id), default=str))

    dlq_items = bulk(dlq_lines)
    dlq_failures = [
        item["index"] for item in dlq_items if item.get("index", {}).get("status", 200) >= 300
    ]
    if dlq_failures:
        # The dead-letter write is the last line of defence. If it fails we must
        # NOT let the batch commit, or those records are gone for good.
        raise RuntimeError(
            f"batch {batch_id}: {len(dlq_failures)} records could not be dead-lettered; "
            f"refusing to commit. First failure: {json.dumps(dlq_failures[0])[:500]}"
        )

    first = rejected[0][1].get("error", {})
    print(
        f"  batch {batch_id}: dead-lettered {len(rejected)} -> {DLQ_INDEX} "
        f"({first.get('type')}: {str(first.get('reason'))[:160]})"
    )
    return len(rejected)


def to_elasticsearch(batch_df, batch_id):
    try:
        _index_batch(batch_df, batch_id)
    except Exception as exc:
        if is_jvm_error(exc):
            reraise_short(batch_id, exc)
        raise


def _index_batch(batch_df, batch_id):
    # Rows are streamed to the driver a partition at a time rather than
    # collect()ed whole, and shipped to _bulk in BULK_MAX_BYTES chunks. A batch
    # is bounded by bytes, not rows: 400 gwalert records is ~400 MB, and a
    # single request that size is refused by Elasticsearch outright.
    indexed = dead_lettered = 0
    chunk, chunk_bytes = [], 0
    for row in batch_df.toLocalIterator():
        chunk.append(row)
        chunk_bytes += len(row["payload"] or "")
        if chunk_bytes >= BULK_MAX_BYTES:
            dead_lettered += index_chunk(chunk, batch_id)
            indexed += len(chunk)
            chunk, chunk_bytes = [], 0
    if chunk:
        dead_lettered += index_chunk(chunk, batch_id)
        indexed += len(chunk)

    BATCH_STATS[batch_id] = {
        "indexed": indexed - dead_lettered,
        "dead_lettered": dead_lettered,
    }


## 5. Run it, watching consumer lag

Runs for `STREAM_SECONDS`, then stops cleanly. The checkpoint under `.checkpoints/gcn-elk`
means a re-run resumes where this one left off; delete that directory to replay from
`earliest`.

"Cleanly" matters: `query.stop()` cancels whatever batch is running, and a cancelled batch
never commits, so everything it read is fetched again next time. `igwn.gwalert` records are
~1 MB and arrive from GCN at only a few hundred KB/s, so a batch of them takes minutes. After
the deadline the loop therefore waits (up to `DRAIN_SECONDS`) for the in-flight batch to
finish before stopping, and `MAX_OFFSETS_PER_TRIGGER` is kept small enough that a batch fits.

Each committed batch prints one line — documents written, how long the batch took, and its lag
(`latestOffset - endOffset` summed per topic, i.e. how many records GCN has that we haven't read
yet). A batch that takes longer than the 10 s trigger is normal while a backlog drains: Spark
starts the next one immediately, and the warning it logs about this is silenced in section 3, the
duration on the line saying the same thing more usefully. **Lag trending upwards is the warning sign that matters**: it
means you are consuming slower than GCN produces, and given a retention window of a few days,
sustained lag eventually turns into permanent loss. Raise `MAX_OFFSETS_PER_TRIGGER` or run for
longer if you see it climbing.

The heartbeat reader runs alongside. Each lag line also reports how long ago the last tick
arrived; a gap longer than `HEARTBEAT_MAX_AGE` is logged to `HEARTBEAT_INDEX` (see section 4)
and the run's closing summary says whether the endpoint was alive at stop.

In [27]:
def source_offsets(progress, field):
    """endOffset / latestOffset arrive as a JSON string: {topic: {partition: offset}}.

    A batch replayed from the offset log on restart reports latestOffset as the
    string "null" (Spark never polled Kafka for it), so parse first, then default.
    """
    raw = progress.get("sources", [{}])[0].get(field)
    if isinstance(raw, str):
        raw = json.loads(raw) if raw.strip() else None
    return raw or {}


def lag_by_topic(progress):
    consumed = source_offsets(progress, "endOffset")
    available = source_offsets(progress, "latestOffset")
    lag = {}
    for topic, partitions in available.items():
        done = consumed.get(topic, {})
        total = 0
        for partition, latest in partitions.items():
            # A partition missing from endOffset hasn't been planned yet; its lag
            # is unknowable from this report, so leave it out rather than guess.
            if partition in done:
                total += max(int(latest) - int(done[partition]), 0)
        if total:
            lag[topic] = total
    return lag


CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

query = (
    decoded.writeStream.foreachBatch(to_elasticsearch)
    .option("checkpointLocation", str(CHECKPOINT_DIR))
    .trigger(processingTime="10 seconds")
    .start()
)

# A throwaway checkpoint: the probe must always begin at `latest`, never resume.
pulse = (
    heartbeat.writeStream.foreachBatch(track_heartbeat)
    .option("checkpointLocation", tempfile.mkdtemp(prefix="gcn-heartbeat-"))
    .trigger(processingTime="10 seconds")
    .start()
)


def stop_when_idle(stream, name, timeout):
    """stop() cancels a running batch outright, and Spark's handling of the
    cancelled Python callback is what produces the StackOverflowError noise at
    shutdown. Wait for the trigger to go idle first; only give up after timeout."""
    deadline = time.time() + timeout
    while stream.isActive and stream.status["isTriggerActive"] and time.time() < deadline:
        time.sleep(0.2)
    if stream.isActive and stream.status["isTriggerActive"]:
        print(f"  ⚠  {name}: still mid-batch after {timeout}s; stopping anyway (it will be replayed)")
    stream.stop()


def heartbeat_note():
    age = heartbeat_age()
    if age is None:
        return "hb --"
    return f"hb {age:.0f}s" + (" STALE" if age > HEARTBEAT_MAX_AGE else "")


def batch_line(progress):
    """One line per committed batch: what it wrote, how long it took, what is left.

    Everything on it is known only after the batch commits, which is why the
    sink records its counts in BATCH_STATS instead of printing them itself --
    two half-reports interleaved with Spark's own logging were the old output.
    """
    batch_id = progress["batchId"]
    stats = BATCH_STATS.pop(batch_id, {})
    dead_lettered = stats.get("dead_lettered", 0)
    docs = f"{stats.get('indexed', 0)} docs" + (f" +{dead_lettered} DLQ" if dead_lettered else "")
    # Duration is the half of "falling behind" that is actually worth knowing:
    # a batch longer than the trigger interval is expected while a backlog
    # drains, and only matters if the lag beside it is not coming down.
    seconds = progress.get("batchDuration", 0) / 1000
    lag = lag_by_topic(progress)
    worst = ", ".join(
        f"{topic}={count}"
        for topic, count in sorted(lag.items(), key=lambda kv: -kv[1])[:3]
    )
    return (
        f"  batch {batch_id:<6}{docs:<16}{seconds:6.1f}s   "
        f"lag {sum(lag.values())}" + (f" ({worst})" if worst else "") + f"   {heartbeat_note()}"
    )


# batchId -> numInputRows, for the closing totals. recentProgress alone would
# undercount a run of more than 100 batches, which is where it is capped.
reported = {}
final_lag = {}
deadline = time.time() + STREAM_SECONDS
try:
    while time.time() < deadline and query.isActive:
        progress = query.lastProgress
        if progress and progress["batchId"] not in reported:
            reported[progress["batchId"]] = progress["sources"][0]["numInputRows"]
            final_lag = lag_by_topic(progress)
            print(batch_line(progress))
        check_heartbeat()
        time.sleep(1)

    # Deadline reached: let the batch in flight finish so its offsets commit.
    # A progress report is only published after a batch commits, so wait for
    # the batch id to advance past the last one reported. (Watching
    # isTriggerActive is not enough: when the query is behind, the next batch
    # starts the instant the previous one ends and the flag never clears.)
    if query.isActive and query.status["isTriggerActive"]:
        committed = (query.lastProgress or {}).get("batchId", -1)
        print(f"  deadline reached mid-batch {committed + 1}; draining (up to {DRAIN_SECONDS}s)...")
        drain = time.time() + DRAIN_SECONDS
        while query.isActive and time.time() < drain:
            progress = query.lastProgress
            if (progress and progress["batchId"] > committed) or not query.status["isTriggerActive"]:
                break
            check_heartbeat()
            time.sleep(0.2)
        else:
            print("  ⚠  batch still running after drain window; stopping anyway (it will be replayed)")
finally:
    # Cancelling the job groups of batches that have already finished is what
    # stop() does here, and DAGScheduler WARNs about each one; see section 3.
    with quiet("org.apache.spark.scheduler.DAGScheduler"):
        # The science query was drained above: its last batch has just committed
        # and the next is still being planned, so stop() lands between batches.
        query.stop()
        # The probe runs ~1s of every 10s trigger, so this almost never waits long.
        stop_when_idle(pulse, "heartbeat", timeout=15)

# The last batch commits after the loop exits, so its progress is only in
# recentProgress; batches older than the 100 it keeps are only in `reported`.
for progress in query.recentProgress:
    reported.setdefault(progress["batchId"], progress["sources"][0]["numInputRows"])

print(f"\nBatches: {len(reported)}")
print(f"Rows: {sum(reported.values())}")

remaining = sum(final_lag.values())
if remaining:
    print(
        f"\n⚠  {remaining} records still unread at stop. They are safe while they remain\n"
        f"   inside GCN's retention window — resume before that expires, or raise\n"
        f"   MAX_OFFSETS_PER_TRIGGER (currently {MAX_OFFSETS_PER_TRIGGER}) to catch up faster."
    )
else:
    print("\nCaught up with all subscribed partitions.")

age = heartbeat_age()
if age is None:
    print(
        f"\n⚠  No heartbeat in {STREAM_SECONDS}s. Either the endpoint is down or the probe never\n"
        f"   connected -- check pulse.exception() and the OAuth credentials."
    )
elif age > HEARTBEAT_MAX_AGE:
    print(f"\n⚠  GCN endpoint stale at stop: last heartbeat {age:.0f}s ago (see {HEARTBEAT_INDEX}).")
else:
    print(f"\nGCN endpoint alive: {HEARTBEAT['ticks']} heartbeats, last {age:.0f}s ago.")

for name, stream in (("stream", query), ("heartbeat", pulse)):
    if stream.exception():
        print(f"{name}: {stream.exception()}")

26/09/18 23:19:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/09/18 23:19:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


  batch 165   0 docs             5.2s   lag 0   hb --


26/09/18 23:19:40 INFO DAGScheduler: Registering RDD 927 (call at /opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py:644) as input to shuffle 126
26/09/18 23:19:40 INFO DAGScheduler: Got map stage job 262 (call at /opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py:644) with 1 output partitions
26/09/18 23:19:40 INFO DAGScheduler: Final stage: ShuffleMapStage 388 (call at /opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py:644)
26/09/18 23:19:40 INFO DAGScheduler: Parents of final stage: List()
26/09/18 23:19:40 INFO DAGScheduler: Missing parents: List()
26/09/18 23:19:40 INFO DAGScheduler: Missing parents found for ShuffleMapStage 388: List()
26/09/18 23:19:40 INFO DAGScheduler: Submitting ShuffleMapStage 388 (MapPartitionsRDD[927] at call at /opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py:644), which h

  batch 166   0 docs             0.9s   lag 0   hb 11s


26/09/18 23:29:22 INFO DAGScheduler: ShuffleMapStage 743 (call at /opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py:644) finished in 977 ms
26/09/18 23:29:22 INFO DAGScheduler: looking for newly runnable stages
26/09/18 23:29:22 INFO DAGScheduler: running: HashSet()
26/09/18 23:29:22 INFO DAGScheduler: waiting: HashSet()
26/09/18 23:29:22 INFO DAGScheduler: failed: HashSet()
26/09/18 23:29:22 INFO DAGScheduler: Got job 500 (call at /opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py:644) with 1 output partitions
26/09/18 23:29:22 INFO DAGScheduler: Final stage: ResultStage 745 (call at /opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py:644)
26/09/18 23:29:22 INFO DAGScheduler: Parents of final stage: List(ShuffleMapStage 744)
26/09/18 23:29:22 INFO DAGScheduler: Missing parents: List()
26/09/18 23:29:22 INFO DAGScheduler: Missing parents found for Re


Batches: 2
Rows: 0

Caught up with all subscribed partitions.

GCN endpoint alive: 592 heartbeats, last 5s ago.


## 6. Verify what landed

In [28]:
es("POST", "/gcn-*/_refresh")

status, cat = es("GET", "/_cat/indices/gcn-*?format=json&h=index,docs.count,store.size")
for row in sorted(cat, key=lambda r: r["index"]):
    print(f"{row['index']:<40} {row['docs.count']:>8} docs  {row['store.size']}")

status, hit = es(
    "GET",
    "/gcn-*/_search",
    {"size": 1, "sort": [{"@timestamp": "desc"}], "query": {"match_all": {}}},
)
hits = hit.get("hits", {}).get("hits", [])
print("\n--- most recent document ---")
print(json.dumps(hits[0]["_source"], indent=2)[:2000] if hits else "no documents yet")

gcn-circulars                                 673 docs  1.7mb
gcn-heartbeat-events                            3 docs  19.8kb
gcn-igwn-gwalert                             2569 docs  2.2gb
gcn-notices-maxi-gsc-known                      8 docs  38.6kb
gcn-notices-maxi-gsc-unknown                    3 docs  48.2kb
gcn-notices-superk-sn-alert                     3 docs  32.9kb

--- most recent document ---
{
  "@timestamp": "2026-09-18T13:28:58.186000+00:00",
  "ingested_at": "2026-09-18T13:29:07.738470+00:00",
  "topic": "igwn.gwalert",
  "kafka": {
    "partition": 0,
    "offset": 108156,
    "key": null
  },
  "alert": {
    "alert_type": "PRELIMINARY",
    "time_created": "2026-09-18T13:28:57Z",
    "superevent_id": "MS260918n",
    "urls": {
      "gracedb": "https://gracedb.ligo.org/v2/superevents/MS260918n/view/"
    },
    "event": {
      "significant": true,
      "time": "2026-09-18T13:28:48.924Z",
      "far": 9.110699364861297e-14,
      "instruments": [
        "H1",
       

## 6b. Offset-gap audit

The real completeness check. Because `_id` is `topic-partition-offset`, every record we ever
consumed occupies a known slot in a contiguous offset sequence. So for each `(topic, partition)`
the arithmetic has to hold:

```
max(kafka.offset) - min(kafka.offset) + 1  ==  number of documents
```

Any shortfall is a record that Kafka had and Elasticsearch does not. The search covers `gcn-*`,
which includes the dead-letter index — quarantined records still count as received, so a
dead-lettered document keeps the arithmetic balanced while still showing up separately below.

One caveat: Kafka offsets are not contiguous *in general* — transaction markers and log
compaction both consume offsets without producing a consumer-visible record. GCN's topics are
plain, non-transactional and non-compacted, so in practice a gap here is a genuine gap. Treat a
non-zero result as a prompt to investigate, not as proof on its own.

In [29]:
es("POST", "/gcn-*/_refresh")

AUDIT = {
    "size": 0,
    "aggs": {
        "by_topic": {
            "terms": {"field": "topic", "size": 100},
            "aggs": {
                "by_partition": {
                    "terms": {"field": "kafka.partition", "size": 200},
                    "aggs": {
                        "min_offset": {"min": {"field": "kafka.offset"}},
                        "max_offset": {"max": {"field": "kafka.offset"}},
                        "docs": {"value_count": {"field": "kafka.offset"}},
                    },
                }
            },
        }
    },
}

status, result = es("GET", "/gcn-*/_search", AUDIT)
if status >= 300:
    raise RuntimeError(f"audit query failed ({status}): {result}")

print(f"{'topic':<34}{'part':>5}{'first':>12}{'last':>12}{'docs':>9}{'missing':>9}")
print("-" * 81)

total_missing = 0
for topic_bucket in result["aggregations"]["by_topic"]["buckets"]:
    for part in topic_bucket["by_partition"]["buckets"]:
        lo = int(part["min_offset"]["value"])
        hi = int(part["max_offset"]["value"])
        docs = int(part["docs"]["value"])
        missing = (hi - lo + 1) - docs
        total_missing += missing
        flag = "  <-- gap" if missing else ""
        print(
            f"{topic_bucket['key']:<34}{part['key']:>5}{lo:>12}{hi:>12}"
            f"{docs:>9}{missing:>9}{flag}"
        )

print("-" * 81)
print(f"total missing across all partitions: {total_missing}")

status, dlq = es("GET", f"/{DLQ_INDEX}/_count")
dlq_count = dlq.get("count", 0) if status < 300 else 0
print(f"dead-lettered (received, not indexed to its topic): {dlq_count}")

if total_missing == 0:
    print("\nNo offset gaps: every record between the first and last consumed is accounted for.")
else:
    print(
        "\nGaps found. Likely causes, in order of probability:\n"
        "  1. FAIL_ON_DATA_LOSS was False on an earlier run and Kafka aged out those offsets.\n"
        "  2. The checkpoint was deleted while the indices were kept, so a range was skipped.\n"
        "  3. Legitimate non-consumer-visible offsets (unlikely on GCN's plain topics)."
    )

topic                              part       first        last     docs  missing
---------------------------------------------------------------------------------
igwn.gwalert                          0      105588      108156     2569        0
gcn.circulars                         0       53147       53819      673        0
gcn.notices.maxi.gsc.known            0           6          13        8        0
gcn.notices.maxi.gsc.unknown          0           2           4        3        0
gcn.notices.superk.sn_alert           0          23          25        3        0
---------------------------------------------------------------------------------
total missing across all partitions: 0
dead-lettered (received, not indexed to its topic): 0

No offset gaps: every record between the first and last consumed is accounted for.


## 6c. Inspect the dead-letter queue

Anything here was received from Kafka intact but refused by Elasticsearch — almost always a
mapping conflict between two notices on the same topic that disagree about a field's type. The
raw payload is preserved verbatim, so once the mapping is fixed these can be replayed straight
out of this index.

In [30]:
status, dlq = es(
    "GET",
    f"/{DLQ_INDEX}/_search",
    {
        "size": 5,
        "sort": [{"@timestamp": "desc"}],
        "aggs": {"by_reason": {"terms": {"field": "error.type", "size": 20}}},
    },
)

if status == 404:
    print(f"{DLQ_INDEX} does not exist yet — nothing has been rejected.")
else:
    hits = dlq["hits"]["hits"]
    print(f"{dlq['hits']['total']['value']} dead-lettered records\n")
    for bucket in dlq.get("aggregations", {}).get("by_reason", {}).get("buckets", []):
        print(f"  {bucket['doc_count']:>6}  {bucket['key']}")
    if hits:
        print("\n--- most recent ---")
        print(json.dumps(hits[0]["_source"], indent=2)[:1500])

gcn-dead-letter does not exist yet — nothing has been rejected.


## 6d. Prune records older than 60 days

Nothing removes documents once they are indexed, so the indices grow without bound. This
deletes documents whose `@timestamp` is older than `RETENTION_DAYS`. (The heartbeat is not
indexed, so the one topic that would have made this urgent contributes nothing.)

Deliberately a `_delete_by_query` rather than an ILM policy: there is one long-lived index per
topic here, and ILM's delete phase acts on *index* age, so it would drop a whole topic in one
go. Document age is what matters.

This does not disturb the offset-gap audit above. Kafka assigns offsets in production order, so
the oldest documents also carry the lowest offsets — pruning removes a contiguous prefix, which
raises `min(kafka.offset)` and reduces the document count by the same amount, leaving
`max - min + 1 == docs` intact.


In [31]:
CUTOFF = f"now-{RETENTION_DAYS}d"
STALE = {"range": {"@timestamp": {"lt": CUTOFF}}}

# Count per index first, so the output says what was removed and from where
# rather than just a total.
status, before = es(
    "GET",
    "/gcn-*/_search",
    {"size": 0, "query": STALE, "aggs": {"by_index": {"terms": {"field": "_index", "size": 50}}}},
)
if status >= 300:
    raise RuntimeError(f"prune survey failed ({status}): {before}")

doomed = {b["key"]: b["doc_count"] for b in before["aggregations"]["by_index"]["buckets"]}

if not doomed:
    print(f"nothing older than {RETENTION_DAYS} days -- nothing to prune")
else:
    for index, count in sorted(doomed.items()):
        print(f"  {index:<40}{count:>9} stale")

    # conflicts=proceed: a document rewritten by an in-flight batch is skipped
    # rather than aborting the whole prune. It will be caught next run.
    status, result = es(
        "POST", "/gcn-*/_delete_by_query?conflicts=proceed&refresh=true", {"query": STALE}
    )
    if status >= 300:
        raise RuntimeError(f"prune failed ({status}): {result}")
    if result.get("failures"):
        raise RuntimeError(f"prune reported failures: {result['failures'][:3]}")

    print(f"\ndeleted {result['deleted']} documents in {result['took']} ms")
    if result.get("version_conflicts"):
        print(f"{result['version_conflicts']} skipped on version conflict; next run retries them")


nothing older than 60 days -- nothing to prune


## 7. Kibana data views + dashboard

A single **GCN master sink** data view over `gcn-*`. Every topic index, the dead-letter index
and the heartbeat-incident index land in the one place, so Discover and every panel below
query the same view and narrow by `topic` (or `_index`) rather than by switching views. Any
leftover per-topic views from earlier runs are removed so there is exactly one.

On top of those, a saved **GCN alerts** dashboard: records per topic over time, the dead-letter
count, a per-topic summary (record count and latest `@timestamp`, which doubles as a quick
"is this topic still arriving?" check), and the most recent circulars. The stored time window is derived from the oldest
`@timestamp` in the indices — the stream reads from `earliest`, so a stock "last 15 minutes"
dashboard would open empty and look broken.

Everything here is keyed by a fixed id, so re-running updates in place instead of piling up
duplicates.


In [32]:
KBN = {"kbn-xsrf": "true"}

status, cat = es("GET", "/_cat/indices/gcn-*?format=json&h=index")
LIVE_INDICES = sorted(row["index"] for row in cat)

status, existing = es("GET", "/api/data_views", base=KIBANA_URL, headers=KBN)
BY_TITLE = {view["title"]: view["id"] for view in existing["data_view"]}

# --- the master sink -------------------------------------------------------
MASTER_VIEW = "gcn-*"
MASTER_NAME = "GCN master sink"

# Never fetch the skymaps into Discover. Each is 1-3 MB of base64 and Discover
# pulls up to 500 hits per page, which overruns Elasticsearch's 10 MB limit on
# an async-search response. They stay in _source for programmatic retrieval.
SOURCE_FILTERS = [
    {"value": "alert.event.skymap"},
    {"value": "alert.external_coinc.combined_skymap"},
    {"value": "payload"},
]

if MASTER_VIEW in BY_TITLE:
    # Same view, possibly created under an older name: bring the name in line.
    MASTER_ID = BY_TITLE[MASTER_VIEW]
    status, body = es(
        "POST",
        f"/api/data_views/data_view/{MASTER_ID}",
        {
            "data_view": {
                "name": MASTER_NAME,
                "timeFieldName": "@timestamp",
                "sourceFilters": SOURCE_FILTERS,
            }
        },
        base=KIBANA_URL,
        headers=KBN,
    )
    if status >= 300:
        raise RuntimeError(f"could not update data view {MASTER_VIEW}: {status} {body}")
else:
    status, body = es(
        "POST",
        "/api/data_views/data_view",
        {
            "data_view": {
                "id": "gcn-master",
                "title": MASTER_VIEW,
                "name": MASTER_NAME,
                "timeFieldName": "@timestamp",
                "sourceFilters": SOURCE_FILTERS,
                # Let the view exist before the first index does, so section 7
                # can run ahead of section 5 without erroring.
                "allowNoIndex": True,
            }
        },
        base=KIBANA_URL,
        headers=KBN,
    )
    if status >= 300:
        raise RuntimeError(f"could not create data view {MASTER_VIEW}: {status} {body}")
    MASTER_ID = body["data_view"]["id"]

# Exactly one view: retire the per-topic ones an earlier revision created.
for title, view_id in list(BY_TITLE.items()):
    if title.startswith("gcn-") and title != MASTER_VIEW:
        status, _ = es("DELETE", f"/api/data_views/data_view/{view_id}", base=KIBANA_URL, headers=KBN)
        print(f"  removed per-topic view {title} ({status})")
        BY_TITLE.pop(title)

print(f"  {MASTER_VIEW:<32} {MASTER_ID}  ({MASTER_NAME})")


# --- panels ----------------------------------------------------------------
COUNT_COLUMN = {
    "label": "Count of records",
    "dataType": "number",
    "operationType": "count",
    "sourceField": "___records___",
    "isBucketed": False,
    "scale": "ratio",
}


def lens_panel(title, visualization_type, columns, column_order, visualization, query=""):
    """A Lens saved object over a single formBased layer of the master view."""
    return {
        "attributes": {
            "title": title,
            "visualizationType": visualization_type,
            "state": {
                "datasourceStates": {
                    "formBased": {"layers": {"l1": {"columnOrder": column_order, "columns": columns}}}
                },
                "filters": [],
                "query": {"language": "kuery", "query": query},
                "visualization": visualization,
            },
        },
        "references": [
            {
                "type": "index-pattern",
                "id": MASTER_ID,
                "name": "indexpattern-datasource-layer-l1",
            }
        ],
    }


records_per_topic = lens_panel(
    "Records per topic",
    "lnsXY",
    {
        "b_time": {
            "label": "@timestamp",
            "dataType": "date",
            "operationType": "date_histogram",
            "sourceField": "@timestamp",
            "isBucketed": True,
            "scale": "interval",
            "params": {"interval": "auto", "includeEmptyRows": True},
        },
        "b_topic": {
            "label": "Topic",
            "dataType": "string",
            "operationType": "terms",
            "sourceField": "topic",
            "isBucketed": True,
            "scale": "ordinal",
            "params": {
                "size": 10,
                "orderBy": {"type": "column", "columnId": "c"},
                "orderDirection": "desc",
                "otherBucket": True,
                "missingBucket": False,
            },
        },
        "c": COUNT_COLUMN,
    },
    ["b_time", "b_topic", "c"],
    {
        "title": "Records per topic",
        "preferredSeriesType": "bar_stacked",
        "legend": {"isVisible": True, "position": "right"},
        "valueLabels": "hide",
        "axisTitlesVisibilitySettings": {"x": False, "yLeft": True, "yRight": True},
        "layers": [
            {
                "layerId": "l1",
                "layerType": "data",
                "seriesType": "bar_stacked",
                "xAccessor": "b_time",
                "splitAccessor": "b_topic",
                "accessors": ["c"],
            }
        ],
    },
)

latest_circulars = lens_panel(
    "Latest circulars",
    "lnsDatatable",
    {
        "b_subject": {
            "label": "Subject",
            "dataType": "string",
            "operationType": "terms",
            "sourceField": "alert.subject",
            "isBucketed": True,
            "scale": "ordinal",
            "params": {
                "size": 15,
                "orderBy": {"type": "column", "columnId": "m_last"},
                "orderDirection": "desc",
                "otherBucket": False,
                "missingBucket": False,
            },
        },
        "m_last": {
            "label": "Received",
            "dataType": "date",
            "operationType": "max",
            "sourceField": "@timestamp",
            "isBucketed": False,
            "scale": "ratio",
        },
    },
    ["b_subject", "m_last"],
    {
        "layerId": "l1",
        "layerType": "data",
        "columns": [{"columnId": "b_subject"}, {"columnId": "m_last"}],
    },
    query='topic : "gcn.circulars"',
)

topic_summary = lens_panel(
    "Topic summary",
    "lnsDatatable",
    {
        "b_topic": {
            "label": "Topic",
            "dataType": "string",
            "operationType": "terms",
            "sourceField": "topic",
            "isBucketed": True,
            "scale": "ordinal",
            "params": {
                "size": 20,
                "orderBy": {"type": "column", "columnId": "c"},
                "orderDirection": "desc",
                "otherBucket": False,
                "missingBucket": False,
            },
        },
        "c": {**COUNT_COLUMN, "label": "Records"},
        "m_last": {
            "label": "Latest record",
            "dataType": "date",
            "operationType": "max",
            "sourceField": "@timestamp",
            "isBucketed": False,
            "scale": "ratio",
        },
    },
    ["b_topic", "c", "m_last"],
    {
        "layerId": "l1",
        "layerType": "data",
        "columns": [{"columnId": "b_topic"}, {"columnId": "c"}, {"columnId": "m_last"}],
    },
)

dead_lettered = lens_panel(
    "Dead-lettered",
    "lnsMetric",
    {"c": COUNT_COLUMN},
    ["c"],
    {"layerId": "l1", "layerType": "data", "metricAccessor": "c"},
    # The dead-letter index may not exist at all, which is the healthy case.
    # Filtering the master view returns 0 rather than erroring.
    query=f'_index : "{DLQ_INDEX}"',
)

PANELS = [
    ("records-per-topic", records_per_topic, {"x": 0, "y": 0, "w": 32, "h": 15}),
    ("dead-lettered", dead_lettered, {"x": 32, "y": 0, "w": 16, "h": 15}),
    ("topic-summary", topic_summary, {"x": 0, "y": 15, "w": 48, "h": 12}),
    ("latest-circulars", latest_circulars, {"x": 0, "y": 27, "w": 48, "h": 15}),
]

panels_json, references = [], []
for slug, panel, grid in PANELS:
    if panel is latest_circulars and "gcn-circulars" not in LIVE_INDICES:
        continue  # nothing indexed for this topic yet
    status, body = es(
        "POST",
        f"/api/saved_objects/lens/gcn-{slug}?overwrite=true",
        panel,
        base=KIBANA_URL,
        headers=KBN,
    )
    if status >= 300:
        raise RuntimeError(f"could not save panel {slug}: {status} {body}")
    panels_json.append(
        {
            "type": "lens",
            "panelIndex": slug,
            "gridData": {**grid, "i": slug},
            "embeddableConfig": {},
            "panelRefName": f"panel_{slug}",
        }
    )
    references.append({"name": f"panel_{slug}", "type": "lens", "id": body["id"]})


# --- dashboard -------------------------------------------------------------
# The stream reads from `earliest`, so @timestamp reaches back as far as GCN's
# retention allows -- a stock "last 15 minutes" dashboard opens empty and looks
# broken. Store a window wide enough to cover what is actually indexed.
status, span = es("GET", "/gcn-*/_search", {"size": 0, "aggs": {"lo": {"min": {"field": "@timestamp"}}}})
oldest = span["aggregations"]["lo"]["value"]
if oldest:
    age = datetime.now(timezone.utc) - datetime.fromtimestamp(oldest / 1000, tz=timezone.utc)
    TIME_FROM = f"now-{age.days + 1}d"
else:
    TIME_FROM = "now-30d"

status, dashboard = es(
    "POST",
    "/api/saved_objects/dashboard/gcn-alerts?overwrite=true",
    {
        "attributes": {
            "title": "GCN alerts",
            "description": "Ingest overview for the GCN -> Elasticsearch pipeline.",
            "panelsJSON": json.dumps(panels_json),
            "optionsJSON": json.dumps({"useMargins": True, "hidePanelTitles": False}),
            "timeRestore": True,
            "timeFrom": TIME_FROM,
            "timeTo": "now",
            "kibanaSavedObjectMeta": {
                "searchSourceJSON": json.dumps({"query": {"language": "kuery", "query": ""}, "filter": []})
            },
        },
        "references": references,
    },
    base=KIBANA_URL,
    headers=KBN,
)
if status >= 300:
    raise RuntimeError(f"could not save dashboard: {status} {dashboard}")

print(f"\n{len(panels_json)} panels, window {TIME_FROM} -> now")
print(f"\nDashboard: {KIBANA_URL}/app/dashboards#/view/{dashboard['id']}")


  gcn-*                            e525922f-b34d-4208-81dd-839f83a71408  (GCN master sink)

4 panels, window now-41d -> now

Dashboard: http://localhost:5601/app/dashboards#/view/gcn-alerts


## 8. Teardown

`down` keeps the `esdata` volume — add `-v` to drop the indexed alerts as well.

In [33]:
TEARDOWN = False

if TEARDOWN:
    down = subprocess.run(
        [DOCKER, "compose", "-f", str(COMPOSE_FILE), "down"],
        capture_output=True,
        text=True,
    )
    print(down.stdout or down.stderr)
else:
    print("set TEARDOWN = True to stop the stack")

set TEARDOWN = True to stop the stack
